# Merge and aggregate resting-state fMRI connectivity matrices
This notebook does the following:
1. Merges preprocessed and preparcellated (into 1000 cortical ROIs following Schaefer 1000 atlas and into 54 subcortical ROIs following Tian 54 atlas) resting-state fMRI time series for all subjects of the depressed and control cohorts defined in 00_inital_cohort_selection.ipynb and 01_cohort_matching.ipynb. This merging results in the creation of a single timeseries file per subject containing the concatenated Schaefer 1000 + Tian 54 timeseries.
2. Handles NaN values in the timeseries data according to specified strategies (e.g., interpolating missing values). Saves this information into CSV files.
3. Determines missing subjects for which timeseries data is unavailable. Saves this information into a CSV file.
4. Computes individual functional connectivity matrices using Pearson correlation.
5. Averages these matrices across subjects to obtain a group-level connectivity matrix. Arithmetic averaging is used by default to reduce memory usage (prior attempts to use numpy stacking and mean caused crashes).
6. Saves individual merged timeseries, individual connectivity matrices, and the group average matrix.
7. Visualizes the average connectivity matrix.
8. Final outputs:
    - `merged_resting_state_timeseries_paths.csv`: CSV file listing paths to merged timeseries files for each subject.
    - `Schaefer7n1000p_TianSubcortexS4_labels.txt`: Text file containing ROI labels for the combined atlas.
    - Per-subject `*_connectivity.npy` matrices: Individual functional connectivity matrices saved as NumPy arrays.
    - `missing_subjects.csv`: CSV file listing subjects missing timeseries data.
    - Per-atlas `*_resting_state_timeseries_nans_info.csv` NaN handling CSV files: Detailing how NaN values were addressed for each subject. 
    - `average_connectivity.npy`: The group average functional connectivity matrix saved as a NumPy array.
    - Per-cohort `*_average_func_conn_matrix_Schaefer7n1000p_TianSubcortexS4.svg` visualization of the average connectivity matrix saved as an image file

## 1) Imports
We import utilities and standard scientific Python libraries.

In [ ]:
from pathlib import Path
import sys

import numpy as np

# Make sure the features directory is importable
PROJECT_ROOT = Path(".../subtyping_depression")
FEATURES_DIR = PROJECT_ROOT / "source_code" / "connectivity_matrices"
if str(FEATURES_DIR) not in sys.path:
    sys.path.insert(0, str(FEATURES_DIR))

from mgng_avg_rest_utils import (
    ConnectivityConfig,
    NaNHandlingConfig,
    prepare_merged_timeseries,
    compute_average_connectivity,
    plot_average_connectivity_matrix,
)

%matplotlib inline

## 2) Configuration
Set the cohort folder, whether to (re)build merged time series, and connectivity parameters.

- By default this notebook uses correlation connectivity (`kind='correlation'`) and arithmetic averaging.
- You can enable additional time-series cleaning via `clean_kwargs` if desired.

In [ ]:
cohort_type = 'F32'  #'F32' or 'control'
data_dir = PROJECT_ROOT / "data" / "UKB" / f"{cohort_type}_notask_STRCO_RSSCHA_RSTIA"
vis_dir = PROJECT_ROOT / "reports" / "figures" / "schaefer1000+tian54" / "functional_con"
vis_dir.mkdir(parents=True, exist_ok=True)

nan_cfg = NaNHandlingConfig(
    interp_method='linear',
    roi_nan_ratio_threshold=0.05,
)

conn_cfg = ConnectivityConfig(
    kind='correlation',
    standardize='zscore_sample',
    clean_kwargs=None,  # set to dict(standardize='zscore_sample', detrend=True, low_pass=0.1, t_r=0.735) if needed
    fisher_z_average=False,
)

save_subject_matrices = True

labels_path = data_dir / "Schaefer7n1000p_TianSubcortexS4_labels.txt"
metadata_paths_csv = data_dir / "merged_resting_state_timeseries_paths.csv"

print("data_dir:", data_dir)
print("vis_dir :", vis_dir)

## 3) Prepare merged time series
Creates merged per-subject `.npy` time series, the metadata CSV, NaN handling CSVs, and missing subjects CSV.

In [ ]:
outputs = prepare_merged_timeseries(data_dir=data_dir, nan_cfg=nan_cfg)
labels_path = outputs['labels_path']
metadata_paths_csv = outputs['metadata_paths_csv']
print('Prepared:')
for k, v in outputs.items():
    print(f'  {k}: {v}')

## 4) Compute average connectivity
This loads each subject's merged time series (excluding those that have missing data), computes a correlation matrix, optionally saves per-subject matrices, and averages across subjects.

In [ ]:
avg_connectivity = compute_average_connectivity(
    metadata_paths_csv=metadata_paths_csv,
    data_dir=data_dir,
    cfg=conn_cfg,
    save_subject_matrices=save_subject_matrices,
)
print('Average matrix shape:', avg_connectivity.shape)
print('Value range:', float(np.min(avg_connectivity)), float(np.max(avg_connectivity)))

## 5) Plot and save average connectivity matrix

In [ ]:
out_svg = vis_dir / f"{cohort_type}_average_func_conn_matrix_Schaefer7n1000p_TianSubcortexS4.svg"
plot_average_connectivity_matrix(
    avg_connectivity=avg_connectivity,
    labels_path=labels_path,
    out_path=out_svg,
)
print('Saved:', out_svg)

## 6) Running the pipeline as a script
If you prefer a non-notebook run, execute `source_code/connectivity_matrices/mgng_avg_rest_main.py` after editing its configuration.